# Cereals GTA version

In [134]:
import numpy as np
import pandas as pd

DATASET_URL = 'https://bit.ly/396WyAZ'
MISSING_COLS = ['sugars', 'potass', 'carbo']
MFR_NAMES = {
    "A": "American Home Food Products",
    "G": "General Mills",
    "K": "Kelloggs",
    "N": "Nabisco",
    "P": "Post",
    "Q": "Quaker Oats",
    "R": "Ralston Purina"
}

TYPE_NAMES = {
    "C": "Cold",
    "H": "Hot"
}

NUTRIENTS = ['calories', 'protein', 'fat', 'sodium', 'fiber', 'carbo', 'sugars', 'potass','vitamins']


def load_dataset(source):
    # load dataset
    df = pd.read_csv(source,
        dtype={
            "calories": "uint8",
            'protein': 'uint8',
            'fat': 'uint8',
            'sodium': 'uint16',
            'fiber': 'float32',
            'carbo': 'float32',
            'sugars': 'float32',
            'potass': 'float32',
            'vitamins': 'uint8',
            'shelf': 'uint8',
            'weight': 'float32',
            'cups': 'float32',
            'rating': 'float64'
        }
    )

    # missing values
    for column in MISSING_COLS:
        df[column] = df[column].replace(-1, np.nan)

    # names mapping
    # df['mfr_name'] = df['mfr'].map(MFR_NAMES)
    # df['type_name'] = df['type'].map(TYPE_NAMES)

    # conversion to 100g
    for nutrient in NUTRIENTS:
        df[f'{nutrient}_100g'] = (df[nutrient] / (df['weight'] * 28.35) * 100).round(1)

    # type conversions

    return df

df = load_dataset(DATASET_URL)
df


,name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,...,rating,calories_100g,protein_100g,fat_100g,sodium_100g,fiber_100g,carbo_100g,sugars_100g,potass_100g,vitamins_100g
0,100% Bran,N,C,70,4,1,130,10.0,5.0,6.0,...,68.402973,246.899994,14.1,3.5,458.600006,35.299999,17.600000,21.200001,987.700012,88.199997
1,100% Natural Bran,Q,C,120,3,5,15,2.0,8.0,8.0,...,33.983679,423.299988,10.6,17.6,52.900002,7.100000,28.200001,28.200001,476.200012,0.000000
2,All-Bran,K,C,70,4,1,260,9.0,7.0,5.0,...,59.425505,246.899994,14.1,3.5,917.099976,31.700001,24.700001,17.600000,1128.699951,88.199997
3,All-Bran with Extra Fiber,K,C,50,4,0,140,14.0,8.0,0.0,...,93.704912,176.399994,14.1,0.0,493.799988,49.400002,28.200001,0.000000,1164.000000,88.199997
4,Almond Delight,R,C,110,2,2,200,1.0,14.0,8.0,...,34.384843,388.000000,7.1,7.1,705.500000,3.500000,49.400002,28.200001,NaN,88.199997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,Triples,G,C,110,2,1,250,0.0,21.0,3.0,...,39.106174,388.000000,7.1,3.5,881.799988,0.000000,74.099998,10.600000,211.600006,88.199997
73,Trix,G,C,110,1,1,140,0.0,13.0,12.0,...,27.753301,388.000000,3.5,3.5,493.799988,0.000000,45.900002,42.299999,88.199997,88.199997
74,Wheat Chex,R,C,100,3,1,230,3.0,17.0,3.0,...,49.787445,352.700012,10.6,3.5,811.299988,10.600000,60.000000,10.600000,405.600006,88.199997
75,Wheaties,G,C,100,3,1,200,3.0,17.0,3.0,...,51.592193,352.700012,10.6,3.5,705.500000,10.600000,60.000000,10.600000,388.000000,88.199997


## Group By

In [135]:
# Aky je priemerny cukor v cerealiach Kellogs?
df.loc[
    df['mfr'] == 'K',
    'sugars_100g'
].mean()

np.float32(24.530436)

In [136]:
results = {}

for name in df['mfr'].unique():
    value = df.loc[
        df['mfr'] == name,
        'sugars_100g'
    ].mean()
    results[name] = value

# print(results)

df['mfr'].unique()

<StringArray>
['N', 'Q', 'K', 'R', 'G', 'P', 'A']
Length: 7, dtype: str

In [137]:
df.groupby('mfr')['sugars_100g'].mean()

# maximum
df.groupby('mfr')['sugars_100g'].max()

# maximum z viacerych stlpcov naraz
df.groupby('mfr')[ ['sugars_100g', 'fiber_100g', 'rating' ] ].mean()

# zoskupenie podla dvoch/viacerych stlpcov
df.groupby( ['mfr', 'type'] )['rating'].mean()

# volanie viacerych funkcii naraz
df.groupby('mfr')['sugars'].agg(['mean', 'max', 'min'])

,mean,max,min
mfr,,,
A,3.000000,3.0,3.0
G,7.954545,14.0,1.0
K,7.565217,15.0,0.0
N,1.833333,6.0,0.0
P,8.777778,15.0,3.0
Q,6.142857,12.0,0.0
R,6.125000,11.0,2.0


In [138]:
# 1. Kolko cerealii ma v datasete kazdy vyrobca? (kto vyraba najviac?)
df.groupby('mfr')['name'].count()
df.groupby('mfr')['name'].max()

df['mfr'].value_counts()


mfr
K    23
G    22
P     9
Q     8
R     8
N     6
A     1
Name: count, dtype: int64

In [139]:
df.loc[
    df['mfr'] == 'R',
     ['mfr', 'name']
]

,mfr,name
4,R,Almond Delight
8,R,Bran Chex
15,R,Corn Chex
23,R,Double Chex
44,R,Muesli Raisins; Dates; & Almonds
45,R,Muesli Raisins; Peaches; & Pecans
61,R,Rice Chex
74,R,Wheat Chex


In [140]:
# 2. ake je priemerne hodnotenie studenych a teplych cerealii?
df.groupby('type')['rating'].mean()


type
C    42.095218
H    56.737708
Name: rating, dtype: float64

In [141]:
# 3. aky je najvyssi obsah cukru na 100 g na kazdej policke?
df.groupby('shelf')['sugars_100g'].max()


shelf
1    52.900002
2    52.900002
3    38.799999
Name: sugars_100g, dtype: float32

In [159]:
# 4. pre kazdeho vyrobcu vypocitajte priemerne, najnizsie a najvyssie hodnotenie. vysledok zoradte podla priemeru.
(
    df.groupby('mfr')['rating']
    .agg(['mean', 'min', 'max', 'count'])
    .sort_values(by='mean', ascending=False)
    .rename(index=MFR_NAMES)
    .rename(columns={
        'mean': 'priemer'
    })
    .round(2)
    # .style.format('{:.2f}')
)

,priemer,min,max,count
mfr,,,,
Nabisco,67.97,59.36,74.47,6
American Home Food Products,54.85,54.85,54.85,1
Kelloggs,44.04,29.92,93.70,23
Quaker Oats,42.92,18.04,63.01,8
Post,41.71,28.03,53.37,9
Ralston Purina,41.54,34.14,49.79,8
General Mills,34.49,19.82,51.59,22


In [148]:
# 5. porovnajte vyrobcov podla priemerneho obsahu cukru, vlakniny a bielkovin na 100g.
df.groupby('mfr')[['sugars_100g','fiber_100g','protein_100g']].mean()


,sugars_100g,fiber_100g,protein_100g
mfr,,,
A,10.600000,0.000000,14.100000
G,26.672728,4.154545,7.822728
K,24.530436,8.939131,8.778261
N,6.466667,14.466667,10.250000
P,28.800001,8.822223,8.111111
Q,21.671429,5.175000,10.575001
R,21.612499,6.612500,8.837500
